In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    roc_curve, 
    accuracy_score,
    f1_score
)
import warnings
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
# from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


In [3]:
base_path = r'/kaggle/input/playground-series-s5e11/'
train_path = base_path + 'train.csv'
test_path = base_path + 'test.csv'

In [5]:
raw_df = pd.read_csv(train_path)

In [6]:
cat = [i for i in raw_df.columns if raw_df[i].dtype == "object"]
target = "loan_paid_back"
num = [i for i in raw_df.columns if i not in (cat + [target, "id"])]
base = [col for col in raw_df.columns if col not in ['id', target]]

print("Categorical Features: ", cat)
print("Numerical Features: ", num)
print("Target Feature: ", [target])
print("Base Features", base)

Categorical Features:  ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
Numerical Features:  ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
Target Feature:  ['loan_paid_back']
Base Features ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']


In [7]:
# Some added Featuure Engineering
# financial feature engineering
def feature_engnineering(raw_df):
    featured_df = raw_df.copy()
    featured_df['credit_utilization'] = featured_df['loan_amount'] / featured_df['annual_income']
    featured_df['interest_burden'] = featured_df['interest_rate'] * featured_df['loan_amount'] / 100
    featured_df['is_employed_flag'] = featured_df['employment_status'].apply(lambda x: 0 if x in ['Unemployed', 'Student', 'Retired'] else 1)
    featured_df['debt_intrest'] = featured_df['debt_to_income_ratio'] * featured_df['interest_rate'] / 100
    featured_df['credit_debt_interaction'] = featured_df['credit_score'] * featured_df['debt_to_income_ratio']
    
    # Example: Tagging high risk
    featured_df['high_risk_combo'] = (
        (featured_df['employment_status'].isin(['Unemployed', 'Student'])) &
        (featured_df['education_level'].isin(['High School', 'Other'])) &
        (featured_df['marital_status'].isin(['Single', 'Divorced', 'Widowed'])) &
        (featured_df['loan_purpose'].isin(['Medical', 'Education', 'Vacation', 'Other'])) &
        (featured_df['grade_subgrade'].str[0].isin(['D', 'E', 'F']))
    )
    return featured_df
    

In [8]:
# After feature engineering
feature_df = feature_engnineering(raw_df)
feature_df.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back,credit_utilization,interest_burden,is_employed_flag,debt_intrest,credit_debt_interaction,high_risk_combo
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0,0.086094,345.635014,1,0.011483,61.824,False
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0,0.207757,593.428520,1,0.021447,105.576,False
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0,0.343080,1659.702640,1,0.009467,67.318,False
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0,0.099929,753.879280,1,0.010465,34.645,False
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0,0.477883,1244.030303,1,0.005411,35.245,False


In [9]:
feature_df.columns

Index(['id', 'annual_income', 'debt_to_income_ratio', 'credit_score',
       'loan_amount', 'interest_rate', 'gender', 'marital_status',
       'education_level', 'employment_status', 'loan_purpose',
       'grade_subgrade', 'loan_paid_back', 'credit_utilization',
       'interest_burden', 'is_employed_flag', 'debt_intrest',
       'credit_debt_interaction', 'high_risk_combo'],
      dtype='object')

In [10]:
feature_num = ['annual_income', 'debt_to_income_ratio', 'credit_score',
               'loan_amount', 'interest_rate', 
               'credit_utilization', 'interest_burden', 'is_employed_flag',
               'debt_intrest', 'credit_debt_interaction', 'high_risk_combo'
              ]
feature_cat = ['gender', 'marital_status', 'education_level', 
               'employment_status', 'loan_purpose', 'grade_subgrade'
              ]
FEATURES = feature_num + feature_cat

In [11]:
train = feature_df.copy()
X = train[FEATURES]
y = train[target]
CATS = feature_cat.copy()
test = feature_engnineering(pd.read_csv(test_path))

# Search For Best Params

In [ ]:
model = lgb.LGBMClassifier(
    objective='binary',
    is_unbalance=True,
    random_state=42,
    n_jobs=-1
)

In [12]:
# =============================
# LightGBM Binary Classification Pipeline (Imbalanced Data)
# =============================

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report

In [13]:
# -----------------------------
# 1. Load or define your data
# -----------------------------
# Example: replace with your dataset
# X = your feature matrix (pd.DataFrame or np.array)
# y = your target (pd.Series or np.array)
# e.g., X, y = df.drop('target', axis=1), df['target']
X[CATS] = X[CATS].astype('category')
# -----------------------------
# 2. Stratified Split
# -----------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print("Train size:", X_train.shape, "Val size:", X_val.shape, "Test size:", X_test.shape)
print("Class balance in train:", np.bincount(y_train))
print("Class balance in val:", np.bincount(y_val))
print("Class balance in test:", np.bincount(y_test))

Train size: (415795, 17) Val size: (89099, 17) Test size: (89100, 17)
Class balance in train: [ 83650 332145]
Class balance in val: [17925 71174]
Class balance in test: [17925 71175]


In [14]:
# -----------------------------
# 3. Compute imbalance ratio
# -----------------------------
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 0.25184783754083306


In [36]:
# -----------------------------
# 4. Define Base Model
# -----------------------------
base_model = lgb.LGBMClassifier(
    device='gpu',
    objective='binary',
    boosting_type='gbdt',
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)


In [18]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


CUDA available: True
Number of GPUs: 1
GPU name: Tesla P100-PCIE-16GB


In [39]:
# -----------------------------
# 5. Define Hyperparameter Space
# -----------------------------
param_grid = {
    'num_leaves': [15, 31, 63, 127],
    'device': ['gpu'],
    'max_depth': [-1, 5, 10, 15],
    'learning_rate': [0.005, 0.01, 0.05, 0.1],
    'n_estimators': [50, 100, 300, 500, 1000],
    'min_child_samples': [10, 20, 50],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 0.5]
}

In [40]:
# -----------------------------
# 6. Stratified Cross-Validation Setup
# -----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [42]:
# -----------------------------
# 7. Hyperparameter Tuning
# -----------------------------
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_grid,
    n_iter=50,
    scoring='roc_auc',
    cv=skf,
    verbose=2,
    random_state=42,
)

random_search.fit(X_train, y_train)

print("\nBest Parameters:")
print(random_search.best_params_)
print("Best CV AUC:", random_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[LightGBM] [Info] Number of positive: 265716, number of negative: 66920
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2358
[LightGBM] [Info] Number of data points in the train set: 332636, number of used features: 17
[LightGBM] [Info] Using GPU Device: Tesla P100-PCIE-16GB, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 14 dense feature groups (5.08 MB) transferred to GPU in 0.008225 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.798819 -> initscore=1.378930
[LightGBM] [Info] Start training from score 1.378930
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

Best Parameters:
{'subsample': 1.0, 'reg_lambda': 0, 'reg_alpha': 0.5, 'num_leaves': 31, 'n_estimators': 300, 'min_child_samples': 20, 'max_depth': 10, 'learning_rate': 0.1, 'device': 'gpu', 'colsample_bytree': 0.6}
Best CV AUC: 0.9213154023202099

In [55]:
# -----------------------------
# 8. Evaluate on Validation Set
# -----------------------------
best_model = random_search.best_estimator_

y_val_prob = best_model.predict_proba(X_val)[:, 1]
threshold = 0.5
print("For:", threshold)
y_val_pred = (y_val_prob > threshold).astype(int)
# threshold = threshold + (i/1000)
print("\nValidation Metrics:")
print("ROC AUC:", roc_auc_score(y_val, y_val_prob))
print("PR AUC:", average_precision_score(y_val, y_val_prob))
print("F1 Score:", f1_score(y_val, y_val_pred))
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

# Beautify Confusion Matrix with labels
conf_matrix = confusion_matrix(y_val,y_val_pred)
df_cm = pd.DataFrame(
    conf_matrix, 
    index=["Actual Negative", "Actual Positive"], 
    columns=["Predicted Negative", "Predicted Positive"]
)
print("Confusion Matrix:")
print(df_cm)
print("*" * 40)
    

For: 0.5

Validation Metrics:
ROC AUC: 0.9202106758697202
PR AUC: 0.974490968830686
F1 Score: 0.9147057972272873

Classification Report:
              precision    recall  f1-score   support

         0.0       0.64      0.79      0.71     17925
         1.0       0.94      0.89      0.91     71174

    accuracy                           0.87     89099
   macro avg       0.79      0.84      0.81     89099
weighted avg       0.88      0.87      0.87     89099

Confusion Matrix:
                 Predicted Negative  Predicted Positive
Actual Negative               14103                3822
Actual Positive                7966               63208
****************************************


In [51]:
# -----------------------------
# 8. Evaluate on Validation Set
# -----------------------------
best_model = random_search.best_estimator_

y_val_prob = best_model.predict_proba(X_val)[:, 1]
threshold = 0.4
for i in range(20):
    print("For:", threshold)
    y_val_pred = (y_val_prob > threshold).astype(int)
    threshold = threshold + (i/1000)
    print("\nValidation Metrics:")
    print("ROC AUC:", roc_auc_score(y_val, y_val_prob))
    print("PR AUC:", average_precision_score(y_val, y_val_prob))
    print("F1 Score:", f1_score(y_val, y_val_pred))
    print("\nClassification Report:")
    print(classification_report(y_val, y_val_pred))
    
    # Beautify Confusion Matrix with labels
    conf_matrix = confusion_matrix(y_val,y_val_pred)
    df_cm = pd.DataFrame(
        conf_matrix, 
        index=["Actual Negative", "Actual Positive"], 
        columns=["Predicted Negative", "Predicted Positive"]
    )
    print("Confusion Matrix:")
    print(df_cm)
    print("*" * 40)
    

For: 0.4

Validation Metrics:
ROC AUC: 0.9202106758697202
PR AUC: 0.974490968830686
F1 Score: 0.9308805093459919

Classification Report:
              precision    recall  f1-score   support

         0.0       0.72      0.74      0.73     17925
         1.0       0.93      0.93      0.93     71174

    accuracy                           0.89     89099
   macro avg       0.83      0.83      0.83     89099
weighted avg       0.89      0.89      0.89     89099

Confusion Matrix:
                 Predicted Negative  Predicted Positive
Actual Negative               13199                4726
Actual Positive                5088               66086
****************************************
For: 0.4

Validation Metrics:
ROC AUC: 0.9202106758697202
PR AUC: 0.974490968830686
F1 Score: 0.9308805093459919

Classification Report:
              precision    recall  f1-score   support

         0.0       0.72      0.74      0.73     17925
         1.0       0.93      0.93      0.93     71174

    accu

In [44]:
# -----------------------------
# 9. Final Evaluation on Test Set
# -----------------------------
y_test_prob = best_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob > 0.5).astype(int)

print("\nTest Metrics:")
print("ROC AUC:", roc_auc_score(y_test, y_test_prob)) 
print("PR AUC:", average_precision_score(y_test, y_test_prob))


Test Metrics:
ROC AUC: 0.9216037693645075
PR AUC: 0.9750091160475465


In [53]:
print("\nBest Parameters:")
print(random_search.best_params_)
print("Best CV AUC:", random_search.best_score_)


Best Parameters:
{'subsample': 1.0, 'reg_lambda': 0, 'reg_alpha': 0.5, 'num_leaves': 31, 'n_estimators': 300, 'min_child_samples': 20, 'max_depth': 10, 'learning_rate': 0.1, 'device': 'gpu', 'colsample_bytree': 0.6}
Best CV AUC: 0.9213154023202099


In [54]:
opt_params = {
    'subsample': 1.0, 
    'reg_lambda': 0, 
    'reg_alpha': 0.5, 
    'num_leaves': 31, 
    'n_estimators': 300, 
    'min_child_samples': 20, 
    'max_depth': 10, 
    'learning_rate': 0.1, 
    'device': 'gpu', 
    'colsample_bytree': 0.6
}

In [59]:
train = feature_df.copy()
X = train[FEATURES]
y = train[target]
CATS = feature_cat.copy()

test = feature_engnineering(pd.read_csv(test_path))

X[CATS] = X[CATS].astype("category")
test[CATS] = test[CATS].astype("category")

In [60]:
best_params = random_search.best_params_

final_model = lgb.LGBMClassifier(
    objective='binary',
    boosting_type='gbdt',
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    **best_params
)

final_model.fit(X,y,eval_metric='auc')

[LightGBM] [Info] Number of positive: 474494, number of negative: 119500
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2357
[LightGBM] [Info] Number of data points in the train set: 593994, number of used features: 17
[LightGBM] [Info] Using GPU Device: Tesla P100-PCIE-16GB, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 14 dense feature groups (9.06 MB) transferred to GPU in 0.011922 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.798820 -> initscore=1.378933
[LightGBM] [Info] Start training from score 1.378933


LGBMClassifier(colsample_bytree=0.6, device='gpu', max_depth=10,
               n_estimators=300, objective='binary', random_state=42,
               reg_alpha=0.5, reg_lambda=0,
               scale_pos_weight=0.25184783754083306)

In [61]:
import joblib

joblib.dump(final_model, "lightgbm_best_model.pkl")

['lightgbm_best_model.pkl']

In [63]:
id_df = test["id"]
test_df = test.drop(columns="id")
ans_df = final_model.predict_proba(test_df)[:, 1]

In [65]:
pd.DataFrame({"id":id_df, "loan_paid_back": ans_df}).to_csv("lgb_opt_ans.csv", index = False)